# Gene Predictability Analysis v3 — Cleaned

**Hypothesis:** Per-gene test performance is largely determined by intrinsic gene properties, not by model capacity limitations.

**v3 changes:**
- Removed redundant `train_nonzero` (fully correlated with `train_coverage`)
- Removed `log_mean`, `log_std`, `snr` (redundant with std/mean/cv)
- Uses precomputed split statistics CSV for speed
- Added feature correlation matrix as sanity check

**Core independent features:**
1. `std_expr` — Expression variability (absolute)
2. `mean_expr` — Average expression level
3. `train_coverage` — Fraction of cells expressing this gene
4. `cv` — Coefficient of variation (std/mean)
5. `tile_coverage_std` — Consistency across test tiles

**Analyses:**
1. Data sizes (split summary)
2. Per-tile test composition
3. Feature correlation matrix (verify no redundancy)
4. Univariate: each property vs test R
5. Top vs bottom gene comparison
6. Random Forest: do properties jointly predict performance?
7. Summary

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score

plt.rcParams.update({
    'figure.dpi':           120,
    'font.size':             10,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'legend.frameon':        False,
})

PROJECT   = Path('/hpc/group/jilab/tc459/MorphPT')
CACHE_DIR = PROJECT / 'cache_crc'

# LoRA multi-task results
EXP_DIR = PROJECT / 'experiments' / 'lora_probing_crc_multi_10.0x_mse'
RESULTS_FILE = EXP_DIR / 'multi_lora_hybrid_results.csv'
SPLITS_FILE = EXP_DIR / 'splits_seed_42.npz'

# Gene stats (pre-existing + newly pre-computed)
GENE_STATS_FILE = CACHE_DIR / 'per_gene' / 'top400_variance_mincov0.1.csv'
SPLIT_STATS_FILE = CACHE_DIR / 'per_gene_split_stats_seed_42.csv'

print('Files to load:')
for f in [RESULTS_FILE, SPLITS_FILE, GENE_STATS_FILE, SPLIT_STATS_FILE]:
    status = '✓' if f.exists() else '✗'
    print(f'  {status} {f}')

print(f'\nIf ✗ on split stats: run precompute_gene_split_stats.py first!')

## 2. Load and merge

In [ ]:
# Load results
df_results = pd.read_csv(RESULTS_FILE)
print(f'Results: {df_results.shape}')

# Load gene stats
df_stats = pd.read_csv(GENE_STATS_FILE)
print(f'Gene stats: {df_stats.shape}')

# Load pre-computed split statistics
df_split = pd.read_csv(SPLIT_STATS_FILE)
print(f'Split stats: {df_split.shape}')

# Merge all three
df = df_results.merge(df_stats, on='gene_name', suffixes=('', '_stats'))
df = df.merge(df_split, on=['gene_idx', 'gene_name'], suffixes=('', '_split'))

# Drop duplicate columns
dup_cols = [c for c in df.columns if c.endswith('_stats') or c.endswith('_split')]
df = df.drop(columns=dup_cols)

# Rename test performance column
if 'test_pearson_s42' in df.columns:
    df['test_r'] = df['test_pearson_s42']
elif 'test_pearson_mean' in df.columns:
    df['test_r'] = df['test_pearson_mean']

# Compute CV if not present
if 'cv' not in df.columns and 'mean_expr' in df.columns and 'std_expr' in df.columns:
    df['cv'] = df['std_expr'] / (df['mean_expr'] + 0.001)

print(f'\nFinal merged: {df.shape}')
print(f'Test r range: {df["test_r"].min():.3f} to {df["test_r"].max():.3f}')
df.head(3)

## 3. Data size summary

In [ ]:
# Load splits for tile info
splits = np.load(SPLITS_FILE, allow_pickle=True)
test_tiles = splits['test_tiles']

print('=' * 60)
print(' SPLIT SIZES')
print('=' * 60)
print(f'  Train: {df["n_train_cells"].iloc[0]:>7,} cells')
print(f'  Val:   {df["n_val_cells"].iloc[0]:>7,} cells')
print(f'  Test:  {df["n_test_cells"].iloc[0]:>7,} cells')
print(f'  Total: {df["n_train_cells"].iloc[0] + df["n_val_cells"].iloc[0] + df["n_test_cells"].iloc[0]:>7,} cells')

print('\n' + '=' * 60)
print(' COVERAGE RANGES (per-gene fraction of cells expressing)')
print('=' * 60)
for split in ['train', 'val', 'test']:
    col = f'{split}_coverage'
    if col in df.columns:
        s = df[col]
        print(f'  {split:>5}: mean={s.mean():.3f}  median={s.median():.3f}  '
              f'min={s.min():.3f}  max={s.max():.3f}')

print('\n' + '=' * 60)
print(' COVERAGE CONSISTENCY ACROSS SPLITS (sanity check)')
print('=' * 60)
print(f'  train vs val coverage: r = {pearsonr(df["train_coverage"], df["val_coverage"])[0]:.4f}')
print(f'  train vs test coverage: r = {pearsonr(df["train_coverage"], df["test_coverage"])[0]:.4f}')

## 4. Per-tile test composition

Test set consists of 4 spatially held-out tiles. Do different tiles differ in size and in gene coverage?

In [ ]:
# Cells per tile
print(f'Test tiles: {test_tiles.tolist()}\n')
print('Cells per tile:')
for t in test_tiles:
    col = f'tile_{int(t)}_n_cells'
    if col in df.columns:
        n = df[col].iloc[0]  # Same for all genes
        pct = 100 * n / df['n_test_cells'].iloc[0]
        print(f'  Tile {int(t):>3}: {n:>6,} cells ({pct:.1f}% of test)')

In [ ]:
# Coverage distribution per tile
cov_cols = [f'tile_{int(t)}_coverage' for t in test_tiles]
cov_cols = [c for c in cov_cols if c in df.columns]

if cov_cols:
    print('Per-tile coverage statistics:\n')
    for col in cov_cols:
        s = df[col]
        print(f'  {col:<25}: mean={s.mean():.3f}  median={s.median():.3f}  '
              f'range=[{s.min():.3f}, {s.max():.3f}]')
    
    # Tile coverage std (measure of tile-to-tile variability)
    print(f'\nPer-gene tile coverage std:')
    s = df['tile_coverage_std']
    print(f'  mean = {s.mean():.4f}, median = {s.median():.4f}, max = {s.max():.4f}')

In [ ]:
# Visual: per-tile coverage distributions
cov_cols = [f'tile_{int(t)}_coverage' for t in test_tiles]
cov_cols = [c for c in cov_cols if c in df.columns]

if cov_cols:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # Boxplot of coverage per tile
    ax = axes[0]
    data = [df[col].values for col in cov_cols]
    labels = [f'Tile {int(t)}' for t in test_tiles]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6,
                    showmeans=True,
                    meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markersize': 5})
    for patch, color in zip(bp['boxes'], ['#2E86AB', '#A23B72', '#F0B441', '#2d6a4f']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Per-gene coverage')
    ax.set_title('Distribution of gene coverage across tiles')
    ax.grid(axis='y', alpha=0.3)
    
    # Scatter: tile coverage std vs test r
    ax = axes[1]
    x = df['tile_coverage_std']
    y = df['test_r']
    ax.scatter(x, y, c=y, cmap='viridis', alpha=0.5, s=15)
    r, _ = pearsonr(x, y)
    ax.set_xlabel('Tile coverage std')
    ax.set_ylabel('Test Pearson r')
    ax.set_title(f'Tile inconsistency vs test r  (Pearson = {r:.3f})')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Feature correlation matrix (verify no redundancy)

Before running RF, make sure our features actually carry different information.

In [ ]:
# Core 5 features (cleaned)
CORE_FEATURES = ['std_expr', 'mean_expr', 'train_coverage', 'cv', 'tile_coverage_std']

# Filter to what's in df
CORE_FEATURES = [c for c in CORE_FEATURES if c in df.columns]
FEATURES_PLUS_Y = CORE_FEATURES + ['test_r']

print(f'Feature correlation matrix ({len(CORE_FEATURES)} features + test_r):\n')
corr_matrix = df[FEATURES_PLUS_Y].corr()
print(corr_matrix.round(3).to_string())

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(8, 6))

corr = df[FEATURES_PLUS_Y].corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# Annotations
for i in range(len(corr)):
    for j in range(len(corr)):
        color = 'white' if abs(corr.iloc[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                color=color, fontsize=9)

ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
ax.set_title('Feature correlation (no redundancy when off-diagonal |r| < 0.9)')

plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
plt.show()

# Check for remaining redundancy
print('\nPairs with |correlation| > 0.9 (excluding diagonal):')
redundant = []
for i in range(len(CORE_FEATURES)):
    for j in range(i+1, len(CORE_FEATURES)):
        r = corr.iloc[i, j]
        if abs(r) > 0.9:
            print(f'  {CORE_FEATURES[i]} <-> {CORE_FEATURES[j]}: r = {r:.3f} (redundant!)')
            redundant.append((CORE_FEATURES[i], CORE_FEATURES[j]))

if not redundant:
    print('  None. All features carry distinct information.')

## 6. Univariate: each property vs test R

In [ ]:
n_props = len(CORE_FEATURES)
n_cols  = min(3, n_props)
n_rows  = int(np.ceil(n_props / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5*n_cols, 4*n_rows),
                          squeeze=False)
axes_flat = axes.flatten()

correlations = []
for i, col in enumerate(CORE_FEATURES):
    ax = axes_flat[i]
    
    x = df[col].values
    y = df['test_r'].values
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    
    ax.scatter(x, y, c=y, cmap='viridis', alpha=0.6, s=15)
    
    p_r, p_p = pearsonr(x, y)
    s_r, _   = spearmanr(x, y)
    
    coef = np.polyfit(x, y, 1)
    x_sorted = np.sort(x)
    ax.plot(x_sorted, np.polyval(coef, x_sorted), 'r-', lw=1, alpha=0.8)
    ax.set_xlabel(col)
    ax.set_ylabel('Test Pearson r')
    ax.set_title(f'{col}\nPearson={p_r:.3f} (p={p_p:.1e})\nSpearman={s_r:.3f}')
    ax.grid(alpha=0.3)
    
    correlations.append({
        'feature':    col, 
        'pearson_r':  p_r, 
        'pearson_p':  p_p, 
        'spearman_r': s_r
    })

for i in range(n_props, len(axes_flat)):
    axes_flat[i].axis('off')

plt.tight_layout()
plt.show()

corr_df = pd.DataFrame(correlations)
print('\nUnivariate correlation summary:')
print(corr_df.round(4).to_string(index=False))

## 7. Top vs bottom genes

In [ ]:
df_sorted = df.sort_values('test_r', ascending=False).reset_index(drop=True)

cols_show = ['gene_name', 'test_r'] + CORE_FEATURES
cols_show = [c for c in cols_show if c in df.columns]

print('=' * 80)
print('TOP 20 BEST-PREDICTED GENES')
print('=' * 80)
print(df_sorted.head(20)[cols_show].round(3).to_string(index=False))

print('\n' + '=' * 80)
print('BOTTOM 20 WORST-PREDICTED GENES')
print('=' * 80)
print(df_sorted.tail(20)[cols_show].round(3).to_string(index=False))

In [ ]:
top50    = df_sorted.head(50)
middle50 = df_sorted.iloc[175:225]
bottom50 = df_sorted.tail(50)

# Compact summary table
comparison = pd.DataFrame({
    'Top 50':    [top50[c].mean() if c in df.columns else np.nan for c in ['test_r'] + CORE_FEATURES],
    'Middle 50': [middle50[c].mean() if c in df.columns else np.nan for c in ['test_r'] + CORE_FEATURES],
    'Bottom 50': [bottom50[c].mean() if c in df.columns else np.nan for c in ['test_r'] + CORE_FEATURES],
}, index=['test_r'] + CORE_FEATURES)

print('Mean values by group:\n')
print(comparison.round(4))

print('\nRatios (Bottom / Top):')
for idx in comparison.index:
    if comparison.loc[idx, 'Top 50'] > 0:
        ratio = comparison.loc[idx, 'Bottom 50'] / comparison.loc[idx, 'Top 50']
        print(f'  {idx:<20}: {ratio:.3f} ({ratio*100:.1f}% of top)')

In [ ]:
# Visual comparison by group
n_props = len(CORE_FEATURES)
fig, axes = plt.subplots(1, n_props, figsize=(3.5*n_props, 4.5))
if n_props == 1:
    axes = [axes]

groups = [('Top 50', top50, '#2d6a4f'),
          ('Middle 50', middle50, '#F0B441'),
          ('Bottom 50', bottom50, '#C44E52')]

for ax, col in zip(axes, CORE_FEATURES):
    positions = np.arange(len(groups))
    data = [g[1][col].values for g in groups]
    labels = [g[0] for g in groups]
    colors = [g[2] for g in groups]
    
    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6,
                    showmeans=True,
                    meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markersize': 5})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(col)
    ax.set_title(col, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Gene properties by predictability group', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Random Forest — joint prediction

Can the 5 core gene properties jointly predict test_r?

In [ ]:
mask = df[CORE_FEATURES + ['test_r']].notna().all(axis=1)
X = df.loc[mask, CORE_FEATURES].values
y = df.loc[mask, 'test_r'].values

print(f'Samples: {len(X)} genes x {X.shape[1]} features')
print(f'Features: {CORE_FEATURES}')

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='r2')

print(f'\n5-fold CV R²:')
print(f'  Folds: {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean:  {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

rf.fit(X, y)
print(f'\nIn-sample R²: {rf.score(X, y):.3f}')

In [ ]:
# Permutation importance
result = permutation_importance(rf, X, y, n_repeats=30, random_state=42, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature':         CORE_FEATURES,
    'importance_mean': result.importances_mean,
    'importance_std':  result.importances_std,
}).sort_values('importance_mean', ascending=False)

print('Permutation importance:')
print(imp_df.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Bar chart of importance
ax = axes[0]
ax.barh(imp_df['feature'][::-1], imp_df['importance_mean'][::-1],
        xerr=imp_df['importance_std'][::-1],
        color='#2E86AB', alpha=0.8, edgecolor='black', lw=0.5)
ax.set_xlabel('Permutation importance')
ax.set_title('Which feature matters most?')
ax.grid(axis='x', alpha=0.3)

# Predicted vs actual
ax = axes[1]
y_pred = rf.predict(X)
ax.scatter(y, y_pred, alpha=0.5, s=15, color='#2E86AB')
lims = [y.min() - 0.02, y.max() + 0.02]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='y = x')
ax.set_xlabel('Actual test r')
ax.set_ylabel('RF prediction (from gene properties)')
ax.set_title(f'Can properties predict test r?\nCV R² = {cv_scores.mean():.3f}')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print('=' * 72)
print(' SUMMARY: Is performance determined by gene properties?')
print('=' * 72)

print(f'\n1. DATA:')
print(f'   Total cells: {df["n_train_cells"].iloc[0] + df["n_val_cells"].iloc[0] + df["n_test_cells"].iloc[0]:,}')
print(f'   Test tiles: {test_tiles.tolist()}')

print(f'\n2. UNIVARIATE (test_r vs each feature):')
for corr in correlations:
    strength = ('STRONG' if abs(corr['pearson_r']) > 0.5
                else 'MODERATE' if abs(corr['pearson_r']) > 0.3
                else 'WEAK')
    print(f'   {corr["feature"]:<25}: r = {corr["pearson_r"]:+.3f} [{strength}]')

print(f'\n3. JOINT PREDICTION (RF using 5 core features):')
print(f'   5-fold CV R²: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

if cv_scores.mean() > 0.5:
    conclusion = 'STRONG EVIDENCE'
elif cv_scores.mean() > 0.3:
    conclusion = 'MODERATE EVIDENCE'
else:
    conclusion = 'WEAK EVIDENCE'
print(f'   -> {conclusion}')

print(f'\n4. TOP vs BOTTOM 50:')
for col in ['test_r'] + CORE_FEATURES:
    if col in df.columns:
        t_val = top50[col].mean()
        b_val = bottom50[col].mean()
        if t_val > 0:
            pct = b_val / t_val * 100
            print(f'   {col:<25}: top={t_val:.3f}  bottom={b_val:.3f}  (bottom is {pct:.0f}% of top)')

# Save
out_path = PROJECT / 'experiments' / 'gene_predictability_analysis_v3.csv'
df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

In [ ]:
# Paper-ready argument
print('=' * 72)
print(' PAPER-READY TEXT')
print('=' * 72)

# Get strongest correlation
best = max(correlations, key=lambda c: abs(c['pearson_r']))
exponent = abs(int(np.log10(max(best['pearson_p'], 1e-300))))

variance_ratio = bottom50['std_expr'].mean() / top50['std_expr'].mean() if top50['std_expr'].mean() > 0 else 0
coverage_ratio = bottom50['train_coverage'].mean() / top50['train_coverage'].mean() if top50['train_coverage'].mean() > 0 else 0

text = f'''
To test whether low-performing genes reflect model limitations or
the intrinsic unpredictability of their expression from H&E images,
we examined the relationship between per-gene test performance and
five intrinsic gene properties: expression variance, mean level,
coverage, coefficient of variation, and per-tile consistency.

Univariate analysis revealed that {best["feature"]} most strongly
correlates with test r (Pearson = {best["pearson_r"]:+.2f}, p < 10^-{exponent}).

A Random Forest model using these 5 features achieved 5-fold CV
R^2 = {cv_scores.mean():.2f} +/- {cv_scores.std():.2f}, indicating that {cv_scores.mean()*100:.0f}%% of the
variance in per-gene predictability can be explained by intrinsic
gene properties alone.

The bottom-50 genes show {variance_ratio*100:.0f}%% of the expression variance and
{coverage_ratio*100:.0f}%% of the coverage of the top-50 genes, consistent with these
being genes whose expression is either rare or uniform across cells,
carrying less morphology-linked signal.

Together, these analyses indicate that the ceiling of per-gene
prediction is largely determined by how much morphology encodes about
each gene, not by model capacity.
'''
print(text)